### 📊 Data Loading, Exploration and Quality Control

In [15]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================
import pandas as pd, numpy as np, seaborn as sns
import matplotlib.pyplot as plt

In [16]:
# ============================================================
# GLOBAL SHARPNESS & ACADEMIC STYLE
# ============================================================ 
plt.rcParams.update({ "figure.dpi": 250,
                      "pdf.fonttype": 42,
                      "font.family": "serif", 
                      "axes.labelsize": 11, 
                      "axes.titlesize": 12, 
                      "axes.titleweight": "bold", 
                      "axes.spines.top": False,
                      "axes.spines.right": False, 
                      "axes.grid": True,
                      "grid.alpha": 0.3, 
                      "grid.linestyle": "--", 
                      "legend.fontsize": 9, 
                      "legend.frameon": True, 
                      "legend.edgecolor": "0.8", 
                      "xtick.labelsize": 10, 
                      "ytick.labelsize": 10, 
                      "lines.linewidth": 1.5, 
                      "savefig.bbox": "tight",
                      "savefig.format": "pdf"})
# Set a professional color palette
sns.set_palette("colorblind")

In [17]:
# ============================================================
# FILE PATHS
# ============================================================
water_file = "/Users/Katia/Desktop/Predicting-Groundwater-Levels-Time-Series-Analysis/daily_groundwater_levels.csv"
climate_file = "/Users/Katia/Desktop/Predicting-Groundwater-Levels-Time-Series-Analysis/precipitation_airtemp_data.csv"

In [18]:
# ============================================================
# LOAD THE TWO DATASETS
# ============================================================

water = pd.read_csv(water_file, skiprows = 1)
climate = pd.read_csv(climate_file, skiprows = 1)


# Display basic information
print("Water dataset - first 5 rows:")
print(water.head())
print(water.shape)
print()

print("Climate dataset - first 5 rows:")
print(climate.head())
print(climate.shape)


print("Water dataset - last 5 rows:")
print(water.tail())


print("Climate dataset - last 5 rows:")
print(climate.tail())


Water dataset - first 5 rows:
         date  water_level
0  2006-01-01        21.52
1  2006-01-02        21.38
2  2006-01-03        21.32
3  2006-01-04        21.29
4  2006-01-05        21.28
(7277, 2)

Climate dataset - first 5 rows:
         date  prcp   tmax   tmin
0  2006-01-01   0.00     46    32
1  2006-01-02   0.32     53    44
2  2006-01-03   0.06     51    42
3  2006-01-04   0.00     52    41
4  2006-01-05   0.02     41    34
(7305, 4)
Water dataset - last 5 rows:
            date  water_level
7272  2025-12-27        21.97
7273  2025-12-28        21.90
7274  2025-12-29        21.99
7275  2025-12-30        22.01
7276  2025-12-31        21.87
Climate dataset - last 5 rows:
            date  prcp   tmax   tmin
7300  2025-12-27   0.00     45    39
7301  2025-12-28   0.25     68    40
7302  2025-12-29   0.11     59    20
7303  2025-12-30   0.01     24    20
7304  2025-12-31   0.08     35    21


In [19]:
# Remove leading/trailing spaces and convert names to lowercase
water.columns = water.columns.str.strip().str.lower()
climate.columns = climate.columns.str.strip().str.lower()

print("\nWater-level columns:")
print(water.columns.tolist())

print("\nClimate columns:")
print(climate.columns.tolist())


Water-level columns:
['date', 'water_level']

Climate columns:
['date', 'prcp', 'tmax', 'tmin']


In [20]:
print('The counts of missing values of the following features are:')
print('- precipitation:', climate['prcp'].isna().sum())
print('- maximum temperature:', climate['tmax'].isna().sum())
print('- minimum temperature:', climate['tmin'].isna().sum())
print('- groundwater level:', water['water_level'].isna().sum())

The counts of missing values of the following features are:
- precipitation: 0
- maximum temperature: 0
- minimum temperature: 0
- groundwater level: 0


In [21]:
print('The missing value counts for all the features show zero!')
print('However, the climate dataset has', climate.shape[0], 'rows, and water dataset has,', water.shape[0], 'rows!')
print('This can only mean there are no NaN values in the water data itself, but there are 28 missing dates relative to the complete daily calendar.')
print('Quickly checking the math: 7305 - 7277 rows = ', climate.shape[0]- water.shape[0])

The missing value counts for all the features show zero!
However, the climate dataset has 7305 rows, and water dataset has, 7277 rows!
This can only mean there are no NaN values in the water data itself, but there are 28 missing dates relative to the complete daily calendar.
Quickly checking the math: 7305 - 7277 rows =  28


So, the groundwater dataset contains no missing values among recorded observations; however, 28 calendar dates are absent from the daily observation sequence. Let's create a simple QC (quality control) table to actually see where we stand on this "missingness".

In [22]:
# ============================================================
# MISSINGNESS AND DATE COMPLETENESS - QUALITY CONTROL TABLE
# ============================================================
expected_dates = pd.date_range(
    start="2006-01-01",
    end="2025-12-31",
    freq="D"
)

# Dates actually present
water_dates = pd.DatetimeIndex(
    water["date"]
)

climate_dates = pd.DatetimeIndex(
    climate["date"]
)

# Missing calendar dates
missing_water_dates = expected_dates.difference(water_dates)
missing_climate_dates = expected_dates.difference(climate_dates)

# QC summary
qc_summary = pd.DataFrame({
    "Dataset": ["Groundwater", "Climate"],
    "Expected Days": [len(expected_dates), len(expected_dates)],
    "Observed Rows": [len(water), len(climate)],
    "Missing Dates": [len(missing_water_dates), len(missing_climate_dates)],
    "Missing Values": [water.isna().sum().sum(), climate.isna().sum().sum()]
})
print(qc_summary)

       Dataset  Expected Days  Observed Rows  Missing Dates  Missing Values
0  Groundwater           7305           7277             28               0
1      Climate           7305           7305              0               0


The QC table shows once again that water dataset has no missing values, but it has 28 missing dates. At this point, I should not do any interpolation, forward filling, backward filling, or dropping of the 28 groundwater observations. That would turn the 28 missing dates into NaNs and then potentially replace them with estimated values. Instead, first I need to characterize the gaps.

In [23]:
# ============================================================
# CONSECUTIVE GROUNDWATER DATA GAPS
# ============================================================

water_dates = pd.DatetimeIndex(
    water["date"].sort_values()
)

full_dates = pd.date_range(
    start=water_dates.min(),
    end=water_dates.max(),
    freq="D"
)

missing_dates = full_dates.difference(
    water_dates
)

# Identify consecutive missing-date groups
missing_series = pd.Series(
    missing_dates,
    index=missing_dates
)

groups = (
    missing_series
    .diff()
    .dt.days
    .ne(1)
    .cumsum()
)

gap_summary = (
    missing_series
    .groupby(groups)
    .agg(
        gap_start="min",
        gap_end="max",
        missing_days="count"
    )
    .reset_index(drop=True)
)

print(gap_summary)

   gap_start    gap_end  missing_days
0 2011-04-25 2011-04-29             5
1 2016-12-28 2016-12-28             1
2 2018-04-23 2018-05-09            17
3 2022-10-24 2022-10-25             2
4 2024-05-15 2024-05-17             3


The gap summary shows exactly what we need to know before modeling. Climate data are temporally complete, while groundwater has 28 missing observations concentrated in five gaps. For a 20-year daily series, that's only *0.38%* of the expected groundwater observations. So this is not a major data-quality problem, but the 17-day gap needs special consideration because it is long enough to potentially affect time-series analysis.